In [1]:
import pandas as pd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mat
import datetime
import time
from tqdm import tqdm

#设置中文字体
mat.rcParams['font.family']='SimHei'
mat.rcParams['font.sans-serif']='SimHei'

#忽略警告
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_read = pd.read_csv(os.getcwd()+"\\cleaned_data\\dacang\\dacang_cleaned_data(indoor including)1104.csv")
df_read

,a,r,t,m
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98"
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53"
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106"
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53"
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106"
...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144"
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144"
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144"
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144"


In [3]:
df = df_read.copy()
df.t = pd.to_datetime(df.t)
df['calender'] = df.t.apply(lambda x : str(x.month)+'.'+str(x.day))
df

,a,r,t,m,calender
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106",7.5
...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144",8.7


In [4]:
#get mac list
mac_list = df.m.unique()
len(mac_list)

5666

In [6]:
#if a MAC stay for more than 8 days and at least 10 hours for each day, this mac will be considered as resident's MAC


def IsTourist(df_now):
    days = df_now.calender.unique()
    count = 0
    for i in range(len(days)):
        df_one = df_now[df_now.calender == days[i]]
        time = df_one.iloc[len(df_one)-1].t - df_one.iloc[0].t
        h = time.components.hours
        if h>10:
            count += 1
    return True if count<8 else False

tourist_list = []
resident_list = []

for mac in tqdm(mac_list,desc="Dividing:"):
    df_now = df[df.m == mac]
    df_now.sort_values(by='t')
    if IsTourist(df_now):
        tourist_list.append(mac)
    else:
        resident_list.append(mac)


Dividing:: 100%|██████████| 5666/5666 [05:40<00:00, 16.63it/s]


In [7]:
print('tourists count:',len(tourist_list))
print('residents count:',len(resident_list))

tourists count: 5516
residents count: 150


In [9]:
df_resident = df[df.m.apply(lambda x : True if x in resident_list else False)]
df_resident

,a,r,t,m,calender
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106",7.5
...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144",8.7


In [10]:
df_tourist = df[df.m.apply(lambda x : True if x in tourist_list else False)]
df_tourist

,a,r,t,m,calender
50,22,-38,2022-07-05 23:52:23,"28,54,187,20,163,246",7.5
55,22,-37,2022-07-05 23:52:49,"28,54,187,20,163,246",7.5
60,22,-41,2022-07-05 23:53:02,"28,54,187,20,163,246",7.5
63,22,-42,2022-07-05 23:53:15,"28,54,187,20,163,246",7.5
65,22,-46,2022-07-05 23:53:28,"28,54,187,20,163,246",7.5
...,...,...,...,...,...
1916061,32,-48,2022-08-07 23:49:21,"124,161,119,201,50,40",8.7
1916067,32,-50,2022-08-07 23:50:13,"124,161,119,201,50,40",8.7
1916075,32,-50,2022-08-07 23:50:52,"124,161,119,201,50,40",8.7
1916081,32,-42,2022-08-07 23:51:44,"124,161,119,201,50,40",8.7


In [12]:
df_resident.to_csv(os.getcwd()+'\\cleaned_data\\dacang\\dacang_resident_data.csv',index=False)
df_tourist.to_csv(os.getcwd()+'\\cleaned_data\\dacang\\dacang_tourist_data.csv',index=False)